# 06 — ECOD Per Symbol
Fit a separate ECOD empirical-CDF detector for each `ExternalSymbol`. Different symbols (INTC, MSFT, GOOG, AMZN, AAPL) have different price scales, spread widths, and size distributions — pooling them dilutes ECOD's tail signal.

Writes `submission_ecod_per_symbol.csv`. Requires notebook 01 to have been run first.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import yaml
from src.features import feature_names
from src.models import score_ecod_per_symbol

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
cols = feature_names(cfg['features']['raw'])

In [ ]:
proc = ROOT / 'data' / 'processed'
train = pd.read_parquet(proc / 'train.parquet')
test = pd.read_parquet(proc / 'test.parquet')
train['ExternalSymbol'].value_counts(), test['ExternalSymbol'].value_counts()

In [ ]:
scores = score_ecod_per_symbol(train, test, cfg['models']['ecod'], cols)
scores.min(), scores.max(), scores.mean()

In [ ]:
out = ROOT / 'outputs' / 'submissions'
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'ID': test['index'], 'TARGET': scores}).to_csv(out / 'submission_ecod_per_symbol.csv', index=False)